<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Find wholesale buyer types

**[Find wholesale buyer types](https://learning.nextia-ai.com/courses/ml/m09/find-wholesale-buyer-types/)** · Machine Learning: From Problem to Reliable Model · Module 9, case study 7 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** cluster 440 real wholesale customers by what they buy, and see why the clustering with the best silhouette is useless here. You fix it with a log and scaling, and draw the customers with PCA. Then you compare the clusters with the channel label the business already uses. Then you test how much the result depends on k, on extreme customers and on the columns.

| | |
|---|---|
| **Time** | About 100 minutes with the lesson page; the notebook runs in under a minute |
| **Needs** | An internet connection for the setup cells. No account. |
| **You should know** | k-means, scaling, silhouette and PCA, from [Unsupervised exploration](https://learning.nextia-ai.com/courses/ml/m07/unsupervised-exploration/). The log before scaling, stability and ARI, from [Segment online-shop customers](https://learning.nextia-ai.com/courses/ml/m09/segment-online-shop-customers/). |
| **You do not need** | The course project. This notebook downloads its own data. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/find-wholesale-buyer-types/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M09-L07-find-wholesale-buyer-types/find-wholesale-buyer-types-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the data

Run the next cell. It makes a folder `find-wholesale-buyer-types` with a `data` folder in it. It downloads `wholesale_customers.csv.gz` (7 KB) from the course's labs repository and checks the file's checksum (SHA-256). Everything this notebook writes goes into `find-wholesale-buyer-types`. It needs an internet connection and no account, and takes a few seconds.

You should see `Ready:` and the file with its size. If you run the cell again, it uses the file that is already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `find-wholesale-buyer-types` (in Colab: the **Files** panel on the left), then run the cell again.

The data is **Wholesale customers** by Margarida Cardoso, from the UCI Machine Learning Repository ([doi.org/10.24432/C5030X](https://doi.org/10.24432/C5030X)), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The course compressed the source file and changed nothing else; the [dataset card](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/case-studies/data/wholesale_customers.md) has the details. It is one year of spending of 440 real customers of a wholesale distributor in Portugal.

In [ ]:
# Setup: download the case-study data into find-wholesale-buyer-types/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/case-studies/data/"
FILES = {
    "wholesale_customers.csv.gz": "c3229cf9d5df7ff307cc554b2a30afa618b0875ba836f1434b09d2d1ca8ea390",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into another project. If the cell ran before, start from the parent.
if Path(".nextia-notebook").exists():
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("find-wholesale-buyer-types").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = WORK / "data" / name
    if not (path.exists() and sha256(path) == expected):
        if (HERE / name).exists():  # a copy next to the notebook
            shutil.copy(HERE / name, path)
        else:
            download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder find-wholesale-buyer-types and run again.")

os.chdir(WORK)  # work in the case-study folder
def size(path):
    n = path.stat().st_size
    return f"{n / 1e6:.1f} MB" if n >= 1e6 else f"{n / 1e3:.0f} KB"

print("Ready:", ", ".join(f"data/{n} ({size(Path('data', n))})" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Inês, the sales manager of a food wholesaler in Portugal, runs two delivery plans. The **kitchen route** brings fresh and frozen food several times a week to hotels, restaurants and cafés (the **Horeca** channel). The **shelf route** brings groceries, milk, detergents and paper once a week to shops (the **Retail** channel). Today the channel label decides the route. Her question: do customers really buy the way their label says? Rui, a sales representative, suggests: "Take the grouping with the best score." Inês and Rui are made up; the data is real. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/find-wholesale-buyer-types/#the-situation) tells the full story.

## 2. The data

Run the next cell. It loads the 440 customers.

> **Check.** You should see `(440, 8) | missing values: 0 | repeated rows: 0`.

In [ ]:
customers = pd.read_csv("data/wholesale_customers.csv.gz")
SPEND = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
print(customers.shape, "| missing values:", customers.isna().sum().sum(),
      "| repeated rows:", customers.duplicated().sum())
customers.head()

**What this tells us.** One row is one customer, with its spending in one year on six kinds of product. The unit is "monetary units" (m.u.). Run the next cell to name the codes of `Channel` and `Region`.

In [ ]:
customers["channel"] = customers["Channel"].map({1: "Horeca", 2: "Retail"})
customers["region"] = customers["Region"].map({1: "Lisbon", 2: "Oporto", 3: "Other"})
pd.crosstab(customers["channel"], customers["region"], margins=True)

> **Check.** You should see 298 Horeca and 142 Retail customers.

**What this tells us.** Some slices are small: 18 shops in Lisbon, 19 in Oporto.

## 3. Frame the problem

The unit is one customer. There is no target. The **channel label stays out of the clustering**: you compare the clusters with it once, at the end, like a test set. Region stays out too, and is used only for a slice check. With 440 customers, every conclusion is tentative.

## 4. Explore what the decision needs

Run the next cell. It shows how the spending is spread, and the skew of each column.

> **Predict.** Which column has the longest tail?

In [ ]:
summary = customers[SPEND].describe().T[["min", "25%", "50%", "75%", "max"]].round(0)
summary["skew"] = customers[SPEND].skew().round(1)
summary

> **Check.** You should see a skew of 11.2 for `Delicassen`.

**What this tells us.** Every column has a long tail, and some customers spend only 3 m.u. a year on a category. The next cell draws groceries as recorded and after a log.

In [ ]:
import matplotlib.pyplot as plt

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.5))
left.hist(customers["Grocery"], bins=19, range=(0, 95000))
left.set_xlabel("Grocery spending (m.u.)"); left.set_ylabel("customers")
right.hist(np.log10(customers["Grocery"]), bins=20, range=(0, 5))
right.set_xlabel("Grocery spending, log10 (3 = 1,000 m.u.)")
plt.show()

Run the next cell. It shows which columns move together.

In [ ]:
customers[SPEND].corr().round(2)

> **Check.** You should see a correlation of 0.92 between `Grocery` and `Detergents_Paper`.

**What this tells us.** Groceries, detergents and paper, and milk move together: the shelf goods. Two strongly linked columns count almost twice in a distance. Section 10 tests this.

## 5. Today's rule: the channel label

The baseline is what the business does today. Run the next cell. It compares the two channels, in amounts and in **shares** (the part of one customer's spending that goes to each category).

In [ ]:
share = customers[SPEND].div(customers[SPEND].sum(axis=1), axis=0)
print("Median spend per year, by channel:")
print(customers.groupby("channel")[SPEND].median())
print("\nMedian share of a customer's spend, by channel:")
print(share.groupby(customers["channel"]).median().round(3))

> **Check.** You should see a median fresh share of 0.492 for Horeca and a median grocery share of 0.339 for Retail.

**What this tells us.** On average, the label works. The question is whether every customer follows the average of its label.

## 6. Rui's shortcut: k-means on the raw spending

Run the next cell. It tries k = 2 to 6 on the spending as recorded.

> **Predict.** Will the silhouette be higher or lower than in the online-shop case study (best 0.456)?

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score


def try_k(X, ks=range(2, 7)):
    """Silhouette and cluster sizes for each k."""
    for k in ks:
        labels = KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(X)
        print(f"k {k}: silhouette {silhouette_score(X, labels):.3f}  "
              f"sizes {sorted(np.bincount(labels).tolist(), reverse=True)}")


try_k(customers[SPEND].to_numpy())

> **Check.** You should see `k 2: silhouette 0.512  sizes [375, 65]`.

The silhouette is high. Run the next cell to see what the two clusters mean.

In [ ]:
raw_labels = KMeans(n_clusters=2, n_init=20, random_state=0).fit_predict(customers[SPEND].to_numpy())
raw_profile = customers.groupby(raw_labels)[SPEND].median()
raw_profile["customers"] = np.bincount(raw_labels)
raw_profile["total"] = customers[SPEND].sum(axis=1).groupby(raw_labels).median()
raw_profile

**What this tells us.** 65 customers who spend a lot on fresh food against everybody else: a split by size, not by buying pattern. Run the next cell: scaling without a log gives an even "better" silhouette.

In [ ]:
from sklearn.preprocessing import FunctionTransformer, StandardScaler

scaled_only = StandardScaler().fit_transform(customers[SPEND])
try_k(scaled_only, ks=[2])

> **Check.** You should see `k 2: silhouette 0.567  sizes [391, 49]`.

**What this tells us.** This is the **silhouette trap**. Silhouette rewards clusters that are far apart, and with a long tail the easiest way is to put the few giants on one side. Decide first what "similar" means for the decision: here, a similar *mix* of goods, at any size. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/find-wholesale-buyer-types/#first-try-k-means-on-the-raw-spending) explains it in full.

## 7. Log and scale

Run the next cell. It takes the log, scales, and tries k = 2 to 6 again.

In [ ]:
from sklearn.pipeline import make_pipeline

prepare = make_pipeline(FunctionTransformer(np.log1p), StandardScaler())
X = prepare.fit_transform(customers[SPEND])
print("skew after the log:", np.log1p(customers[SPEND]).skew().round(1).to_dict())
try_k(X)

> **Check.** You should see `k 2: silhouette 0.290  sizes [252, 188]`.

**What this tells us.** The silhouette is lower, but the clusters have sizes that could be real types. Real buyers overlap; the method no longer cheats with the giants.

Run the next cell. It checks the stability of each k with 20 fits on random 80% samples. It takes about 10 seconds.

In [ ]:
def stability(X, k, runs=20):
    """Mean agreement (ARI) between k-means runs fitted on different random 80% samples."""
    rng = np.random.default_rng(0)
    labels = []
    for run in range(runs):
        rows = rng.choice(len(X), int(0.8 * len(X)), replace=False)
        labels.append(KMeans(n_clusters=k, n_init=20, random_state=run).fit(X[rows]).predict(X))
    return np.mean([adjusted_rand_score(a, b) for i, a in enumerate(labels) for b in labels[i + 1:]])


for k in range(2, 7):
    print(f"k {k}: stability {stability(X, k):.3f}")

> **Check.** You should see `k 2: stability 0.921` and lower values for every larger k.

**What this tells us.** Only k = 2 is stable: 440 customers are not enough for finer types. You choose k = 2.

## 8. The two buyer types

Run the next cell. It fits the pipeline with k = 2 and names the cluster with the higher median grocery spending "shelf", the other "kitchen".

In [ ]:
segmenter = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                          KMeans(n_clusters=2, n_init=20, random_state=0))
customers["cluster"] = segmenter.fit_predict(customers[SPEND])
shelf = customers.groupby("cluster")["Grocery"].median().idxmax()
customers["buyer_type"] = np.where(customers["cluster"] == shelf, "shelf", "kitchen")
profile = customers.groupby("buyer_type")[SPEND].median()
profile["customers"] = customers["buyer_type"].value_counts()
profile

> **Check.** You should see 252 kitchen buyers and 188 shelf buyers.

**What this tells us.** Kitchen buyers spend about twice as much on fresh and frozen food. Shelf buyers spend about four times as much on groceries and milk, and more than ten times as much on detergents and paper. The clustering found two kinds of business without seeing the label.

### A picture: PCA

Run the next cell. It fits PCA and shows what each direction means.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(X)
print("share of the spread kept:", pca.explained_variance_ratio_.round(3))
pd.DataFrame(pca.components_, index=["PC1", "PC2"], columns=SPEND).round(2)

> **Check.** You should see `[0.441 0.272]`, and PC1 with weights of about 0.55 on milk, groceries and detergents and paper.

**What this tells us.** PC1 is "shelf goods", PC2 is "kitchen goods". The picture keeps 71.3% of the spread. The next cell draws the customers, coloured by label and type.

In [ ]:
xy = pca.transform(X)
fig, ax = plt.subplots(figsize=(7, 5))
for group in ["Horeca / kitchen", "Retail / shelf", "Horeca / shelf", "Retail / kitchen"]:
    mine = (customers["channel"] + " / " + customers["buyer_type"]).to_numpy() == group
    ax.scatter(xy[mine, 0], xy[mine, 1], s=14, label=f"{group} ({mine.sum()})")
ax.set_xlabel("PC1: more milk, groceries, detergents and paper")
ax.set_ylabel("PC2: more fresh, frozen, delicatessen")
ax.legend()
plt.show()

## 9. Compare with the channel label

Now open the sealed label, once.

> **Predict.** How many Horeca customers will be shelf buyers?

In [ ]:
print(pd.crosstab(customers["buyer_type"], customers["channel"], margins=True))
print("ARI with the channel:", round(adjusted_rand_score(customers["channel"], customers["buyer_type"]), 3))

> **Check.** You should see 54 Horeca shelf buyers and `ARI with the channel: 0.514`.

> **Your turn.** How many Retail customers are kitchen buyers? Compute it from `customers` and write it into `retail_kitchen`, then run the check.

In [ ]:
retail_kitchen = int(((customers["channel"] == "Retail") & (customers["buyer_type"] == "kitchen")).sum())
retail_kitchen

In [ ]:
expect_hash('the number of Retail customers who buy like kitchens', retail_kitchen, '2c624232cdd22177')

### The customers who buy unlike their label

Run the next cell. It compares the four groups of label and type by their shares of spending.

In [ ]:
customers["group"] = customers["channel"] + " / " + customers["buyer_type"]
groups = pd.concat([customers["group"].value_counts().rename("customers"),
                    customers[SPEND].sum(axis=1).groupby(customers["group"]).median().rename("total"),
                    share.groupby(customers["group"]).median().round(2)], axis=1)
groups.sort_values("customers", ascending=False)

> **Check.** You should see 54 customers in `Horeca / shelf`, with a fresh share of 0.19, against 0.53 for `Horeca / kitchen`.

**What this tells us.** 54 hotels, restaurants and cafés buy like shops. This is the business finding: today they are all on the kitchen route.

> **Your turn.** What share of **all** spending (all six columns, all 440 customers) comes from the 54 `Horeca / shelf` customers? Round it to 3 decimals with `round(..., 3)` and write it into `share_54`.

In [ ]:
horeca_shelf = customers["group"] == "Horeca / shelf"
share_54 = round(float(customers.loc[horeca_shelf, SPEND].sum().sum() / customers[SPEND].sum().sum()), 3)
share_54

In [ ]:
expect_hash('the share of all spending from the 54 customers', share_54, '25bb420c0f2c47a2')

Run the next cell. It counts the groups in each region.

In [ ]:
pd.crosstab(customers["group"], customers["region"], margins=True)

**What this tells us.** Shop-like Horeca customers are in every region. The Oporto count (2 of 28) is too small to read.

## 10. How much does the result depend on the choices?

The label is open now, so you can look back at how the agreement changes with k. Run the next cell.

In [ ]:
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(X)
    print(f"k {k}: ARI with the channel {adjusted_rand_score(customers['channel'], labels):.3f}  "
          f"Retail share per cluster {pd.Series(customers['Channel'] == 2).groupby(labels).mean().round(2).tolist()}")
print(f"raw spend, k 2: ARI with the channel {adjusted_rand_score(customers['channel'], raw_labels):.3f}")

> **Check.** You should see the ARI fall from 0.514 at k = 2 to 0.200 at k = 6, and an ARI of −0.031 for the raw clustering.

Run the next cell to see what the third cluster of k = 3 adds.

In [ ]:
three = KMeans(n_clusters=3, n_init=20, random_state=0).fit_predict(X)
k3_profile = customers.groupby(three)[SPEND].median()
k3_profile["customers"] = np.bincount(three)
k3_profile["Retail"] = pd.crosstab(three, customers["channel"])["Retail"]
k3_profile

**What this tells us.** The kitchen buyers stay (212, only 3 shops). The shelf side splits into 81 customers with almost no fresh food and 147 large all-round buyers. That is a size split, not a new delivery plan.

Run the next cell. It finds the customers who are more than 3 standard deviations from the mean after the log, and clusters again without them.

In [ ]:
extreme = (np.abs(X) > 3).any(axis=1)
print(f"{extreme.sum()} customers have a value more than 3 SDs from the mean, after the log")
print("their smallest yearly spend in one category:",
      sorted(customers.loc[extreme, SPEND].min(axis=1).tolist()))
refit = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                      KMeans(n_clusters=2, n_init=20, random_state=0))
without = refit.fit_predict(customers.loc[~extreme, SPEND])
print("agreement (ARI) with the clusters of all 440, on the other",
      (~extreme).sum(), "customers:", round(adjusted_rand_score(customers.loc[~extreme, "cluster"], without), 3))

> **Check.** You should see 18 customers, and an agreement of 0.972 on the other 422.

**What this tells us.** After the log, the extreme customers are those who spend almost nothing on one category (eight spent 3 m.u.). The result does not depend on them.

### Change one thing: one column less

> **Your turn.** `Grocery` and `Detergents_Paper` move together (0.92). Cluster on the five other columns. Before you run it, predict: will more or fewer Horeca customers become shelf buyers? Write the count into `horeca_shelf_five`.

In [ ]:
five = [c for c in SPEND if c != "Detergents_Paper"]
variant = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                        KMeans(n_clusters=2, n_init=20, random_state=0))
labels = variant.fit_predict(customers[five])
shelf_five = customers.groupby(labels)["Grocery"].median().idxmax()
type_five = np.where(labels == shelf_five, "shelf", "kitchen")
print("ARI with the channel:", round(adjusted_rand_score(customers["channel"], type_five), 3))
horeca_shelf_five = int(((customers["channel"] == "Horeca") & (type_five == "shelf")).sum())
print("Horeca customers of the shelf type:", horeca_shelf_five)

In [ ]:
expect_hash('the Horeca shelf buyers without Detergents_Paper', horeca_shelf_five, '8722616204217edd')

Compare with the 54 of section 9. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/find-wholesale-buyer-types/#your-turn-one-column-less) explains the result.

## 11. The experiment log

Run the next cell. It writes `experiment_log.csv`, including the rejected options with the highest silhouettes.

In [ ]:
from datetime import date

log = pd.DataFrame([
    ("E01", "Today's rule: the channel label", None, None, "Horeca 298, Retail 142", "baseline", "What the sales team uses now."),
    ("E02", "k-means on raw spend, k=2", 0.512, None, "sizes 375 / 65", "reject", "Splits off big Fresh buyers; not a buying pattern."),
    ("E03", "k-means, scaled only, k=2", 0.567, None, "sizes 391 / 49", "reject", "Splits off the largest customers."),
    ("E04", "k-means, log + scale, k=2", 0.290, 0.921, "kitchen 252 / shelf 188", "keep", "Two buying patterns a manager can use."),
    ("E05", "k-means, log + scale, k=3", 0.259, None, "adds 147 large all-round buyers", "reject", "A size split; no new delivery plan."),
    ("E06", "Check: buyer type against channel", None, None, "ARI 0.514; 54 Horeca buy like shops", "keep", "The finding for the sales team."),
], columns=["id", "change", "silhouette", "stability", "result", "decision", "reason"])
log.insert(1, "date", date.today().isoformat())
log.to_csv("experiment_log.csv", index=False)
log[["id", "change", "decision"]]

## 12. Hand over the list

Run the next cell. It writes `buyer_types.csv`: every customer with its type, and the 62 who buy unlike their label marked for review.

> **Check.** You should see 62 customers marked `buys unlike its channel`.

In [ ]:
deliverable = customers[["channel", "region", *SPEND, "buyer_type"]].copy()
deliverable["review"] = np.where(customers["group"].isin(["Horeca / shelf", "Retail / kitchen"]),
                                 "buys unlike its channel", "")
deliverable.to_csv("buyer_types.csv", index_label="customer")
print(deliverable["review"].value_counts().to_string())

## 13. Recommendations, limits and your portfolio

The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/find-wholesale-buyer-types/#recommendations) has the recommendations for Inês and the limits. With 440 customers, one year and no dates, the result is a list to check, not a decision. The page also has a guide to turn this notebook into a portfolio project. The two files are in the folder `find-wholesale-buyer-types`.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m09/find-wholesale-buyer-types/#knowledge-check).

In this case study you saw the silhouette trap: on raw spending the best silhouette (0.512) only separated the largest customers. After a log and scaling, two stable buyer types (kitchen 252, shelf 188) agreed with the channel label (ARI 0.514). 54 Horeca customers turned out to buy like shops. With 440 customers, the result is a list for the sales team to check. Next: the final assignment of the course.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/ml/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m09/find-wholesale-buyer-types/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real (UCI Wholesale customers, CC BY 4.0); the people in the story are made up.